# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [3]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [5]:
# Task 1 — Multi-series line with highlight
# YOUR CODE HERE

# Step 1: Filter to Asian countries
asia_df = df[df['Region'] == 'Asia'].sort_values(['Country', 'Year'])

print("Asian countries:", asia_df['Country'].unique())
print(asia_df)

# Step 2: Create the figure
fig1 = go.Figure()

# Step 3: Get unique countries
countries = asia_df['Country'].unique()
highlight_country = 'China'  # YOU CAN CHANGE THIS!

# Step 4: Add each country as a separate line
for country in countries:
    country_data = asia_df[asia_df['Country'] == country]

    if country == highlight_country:
        # Highlighted line: bold, colored
        fig1.add_trace(go.Scatter(
            x=country_data['Year'],
            y=country_data['CO2_Mt'],
            mode='lines+markers',
            name=country,
            line=dict(color='#E74C3C', width=4),  # Red, bold
            marker=dict(size=8),
            hovertemplate=f'<b>{country}</b><br>Year: %{{x}}<br>CO2: %{{y:.1f}} Mt<extra></extra>'
        ))
    else:
        # Background lines: grey, thin
        fig1.add_trace(go.Scatter(
            x=country_data['Year'],
            y=country_data['CO2_Mt'],
            mode='lines',
            name=country,
            line=dict(color='#DDDDDD', width=1.5),  # Light grey
            hovertemplate=f'<b>{country}</b><br>Year: %{{x}}<br>CO2: %{{y:.1f}} Mt<extra></extra>',
            showlegend=False  # Don't show grey lines in legend
        ))

# Step 5: Add direct label for highlighted country (instead of legend)
# Find the last point of the highlighted country
last_row = asia_df[asia_df['Country'] == highlight_country].iloc[-1]
fig1.add_annotation(
    x=last_row['Year'],
    y=last_row['CO2_Mt'],
    text=f"  <b>{highlight_country}</b>",
    showarrow=False,
    xanchor='left',
    font=dict(size=12, color='#E74C3C')
)

# Step 6: Polish the layout
fig1.update_layout(
    title={
        'text': '<b>China\'s CO2 Trajectory: The Asian Emissions Powerhouse</b><br><sub>While Asian peers rise modestly, China\'s emissions tripled in two decades</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18, 'color': '#2C3E50'}
    },
    xaxis_title='<b>Year</b>',
    yaxis_title='<b>CO2 Emissions (Million Tonnes)</b>',
    xaxis=dict(
        gridcolor='#ECF0F1',
        showgrid=True,
        zeroline=False
    ),
    yaxis=dict(
        gridcolor='#ECF0F1',
        showgrid=True,
        zeroline=False
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    hovermode='x unified',
    margin=dict(l=80, r=150, t=120, b=80),
    font=dict(family='Arial', size=12, color='#2C3E50'),
    height=500,
    width=1000,
    showlegend=False
)

fig1.show()

Asian countries: ['China' 'India' 'Indonesia' 'Japan' 'South Korea']
         Country Region  Year  CO2_Mt  CO2_per_capita
23         China   Asia  2000  3513.8            1.32
24         China   Asia  2001  3951.7            1.30
25         China   Asia  2002  4312.0            1.45
26         China   Asia  2003  4691.9            1.64
27         China   Asia  2004  5078.9            1.73
..           ...    ...   ...     ...             ...
248  South Korea   Asia  2018   738.5            2.21
249  South Korea   Asia  2019   750.8            2.05
250  South Korea   Asia  2020   755.8            2.19
251  South Korea   Asia  2021   777.3            2.04
252  South Korea   Asia  2022   789.5            2.15

[115 rows x 5 columns]


---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [8]:
# Task 2 — Slopegraph: regional averages
# YOUR CODE HERE
# Task 2: Slopegraph - Regional Change 2000 vs 2022 (FIXED)
# =========================================================

# Step 1: Aggregate to regional averages by year
regional_avg = df.groupby(['Region', 'Year'])['CO2_Mt'].mean().reset_index()

# Step 2: Pivot to get 2000 and 2022 in separate columns
regional_wide = regional_avg.pivot(index='Region', columns='Year', values='CO2_Mt')
regional_wide['Change'] = regional_wide[2022] - regional_wide[2000]

# Sort by 2000 value (so labels don't overlap)
regional_wide = regional_wide.sort_values(2000, ascending=False)

print("Regional change:")
print(regional_wide)

# Step 3: Create the slopegraph
fig2 = go.Figure()

# Step 4: Add lines for each region
for idx, region in enumerate(regional_wide.index):
    value_2000 = regional_wide.loc[region, 2000]
    value_2022 = regional_wide.loc[region, 2022]
    change = value_2022 - value_2000

    # Determine color based on direction
    if change > 0:
        line_color = '#E74C3C'  # Red for increase
    else:
        line_color = '#27AE60'  # Green for decrease

    # Add line connecting 2000 to 2022
    fig2.add_trace(go.Scatter(
        x=[2000, 2022],
        y=[value_2000, value_2022],
        mode='lines',
        name=region,
        line=dict(color=line_color, width=3),
        hovertemplate=f'<b>{region}</b><br>Year: %{{x}}<br>CO2: %{{y:.2f}} Mt<extra></extra>',
        showlegend=False
    ))

    # FIXED: Label at 2000 with vertical offset to prevent overlap
    fig2.add_annotation(
        x=1999.5,
        y=value_2000,  # Use actual value
        text=f'<b>{region}</b><br>{value_2000:.2f}',
        showarrow=False,
        xanchor='right',
        yanchor='middle',  # ADD THIS
        font=dict(size=11, color=line_color)
    )

    # FIXED: Label at 2022 with vertical offset
    fig2.add_annotation(
        x=2022.5,
        y=value_2022,  # Use actual value
        text=f'{value_2022:.2f}',
        showarrow=False,
        xanchor='left',
        yanchor='middle',  # ADD THIS
        font=dict(size=11, color=line_color, weight='bold')
    )

# Step 5: Add custom legend (invisible traces for legend)
fig2.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    name='Increased',
    line=dict(color='#E74C3C', width=3)
))
fig2.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    name='Decreased',
    line=dict(color='#27AE60', width=3)
))

# Step 6: Update layout
fig2.update_layout(
    title={
        'text': '<b>The Great Divergence: Which Regions Are Cleaning Up?</b><br><sub>Most regions increased emissions from 2000-2022; few show decline</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18, 'color': '#2C3E50'}
    },
    xaxis=dict(
        tickvals=[2000, 2022],
        ticktext=['2000', '2022'],
        gridcolor='#ECF0F1',
        zeroline=False,
        range=[1998, 2024]
    ),
    yaxis=dict(
        title='<b>Average CO2 Emissions (Million Tonnes)</b>',
        gridcolor='#ECF0F1',
        zeroline=False
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    hovermode='closest',
    margin=dict(l=150, r=150, t=120, b=80),
    font=dict(family='Arial', size=12, color='#2C3E50'),
    height=600,
    width=1000,
    legend=dict(
        x=1.1,
        y=1,
        xanchor='left',
        yanchor='top'
    )
)

fig2.show()

Regional change:
Year               2000      2001      2002      2003      2004      2005  \
Region                                                                      
North America  3223.600  3154.200  3118.950  3079.000  3153.500  3071.250   
Asia           1280.400  1405.560  1484.960  1582.860  1682.520  1784.920   
Europe          834.925   820.325   813.225   808.725   772.325   753.675   
Africa          425.400   430.000   427.000   456.800   455.000   457.700   
Latin America   419.850   437.100   439.700   448.200   448.500   468.400   
Oceania         378.400   391.500   388.800   394.900   397.800   407.500   

Year               2006     2007     2008      2009  ...      2014     2015  \
Region                                               ...                      
North America  2963.000  3006.10  2950.75  2791.150  ...  2705.850  2705.70   
Asia           1908.660  2007.92  2084.32  2187.860  ...  2693.160  2796.04   
Europe          736.625   711.80   711.35   697.67